# InjectionParams — Parquet Read Performance

Interactive notebook for exploring read and query performance of `InjectionParams`
data written to Parquet by `write_parquet.py`.

Three dataset sizes are available: **10 k**, **50 k**, **100 k**, **1 M** mixed WNB / SG / BBH rows.

**Workflow**

1. Setup & data check  
2. Inspect a file — schema, row counts, approximant split  
3. Timed read benchmarks (full scan + column selection)  
4. Query benchmarks — typed columns vs JSON blob fields, PyArrow vs DuckDB  
5. Visualise results with interactive plots

In [10]:
%matplotlib inline
# ── 1. Imports & path setup ────────────────────────────────────────────────
import json, os, sys, time
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pyarrow as pa
import pyarrow.compute as pc
import pyarrow.parquet as pq

# Add workspace root so pycwb is importable without a full pip install
_HERE = Path(os.path.abspath(""))   # notebook working directory
_ROOT = _HERE.parents[1]
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from pycwb.types.trigger import InjectionParams

try:
    import duckdb
    HAS_DUCKDB = True
    print("DuckDB available:", duckdb.__version__)
except ImportError:
    HAS_DUCKDB = False
    print("DuckDB not found — install with: pip install duckdb")

PARQUET_DIR = _HERE / "parquet"
SIZES       = ["10k", "50k", "100k", "1000k"]
FILES       = {s: PARQUET_DIR / f"params_{s}.parquet" for s in SIZES}

missing = [s for s, p in FILES.items() if not p.exists()]
if missing:
    print(f"⚠  Missing Parquet files for sizes: {missing}")
    print("   Run:  python generate_params.py && python write_parquet.py")
else:
    for s, p in FILES.items():
        mb = p.stat().st_size / 1e6
        print(f"  params_{s}.parquet  {pq.read_metadata(p).num_rows:>8,} rows  {mb:.2f} MB")

DuckDB available: 1.5.0
  params_10k.parquet    10,000 rows  2.36 MB
  params_50k.parquet    50,000 rows  11.79 MB
  params_100k.parquet   100,000 rows  23.58 MB
  params_1000k.parquet  1,000,000 rows  235.78 MB


## 2 · Schema & approximant distribution

In [11]:
# ── 2a. Schema inspection ──────────────────────────────────────────────────
size_sel = widgets.ToggleButtons(
    options=SIZES, value="10k",
    description="Dataset:",
    button_style="info",
)
display(size_sel)

schema_out = widgets.Output()
display(schema_out)

def _show_schema(change=None):
    with schema_out:
        clear_output(wait=True)
        path = FILES[size_sel.value]
        meta = pq.read_metadata(path)
        schema = pq.read_schema(path)
        print(f"File   : {path.name}")
        print(f"Rows   : {meta.num_rows:,}")
        print(f"Groups : {meta.num_row_groups}")
        print(f"\nTop-level fields ({len(schema)} total):")
        for i, field in enumerate(schema):
            print(f"  [{i}] {field.name}: {field.type}")

size_sel.observe(_show_schema, names="value")
_show_schema()

ToggleButtons(button_style='info', description='Dataset:', options=('10k', '50k', '100k', '1000k'), value='10k…

Output()

In [12]:
# ── 2b. Approximant distribution ────────────────────────────────────────────
def approximant_counts(path: Path) -> dict[str, int]:
    tbl    = pq.read_table(path, columns=["injection"])
    col    = tbl.column("injection")
    approx = pc.struct_field(col, "approximant")
    unique = approx.unique().to_pylist()
    return {
        a: int(pc.sum(pc.equal(approx, a).cast(pa.int32())).as_py())
        for a in sorted(unique)
    }

dist_out = widgets.Output()
display(dist_out)

def _show_dist(change=None):
    with dist_out:
        clear_output(wait=True)
        counts = approximant_counts(FILES[size_sel.value])
        total  = sum(counts.values())
        print(f"\nApproximant distribution ({total:,} rows total):")
        for approx, n in sorted(counts.items(), key=lambda x: -x[1]):
            bar = "█" * int(40 * n / total)
            print(f"  {approx:<22} {n:>7,}  {bar}")

size_sel.observe(_show_dist, names="value")
_show_dist()

Output()

## 3 · Full read benchmarks

Measure wall-clock time for reading the whole file with different column selections.
Click **▶ Run read benchmarks** to populate the timing table.

In [13]:
# ── 3. Timed full-read benchmarks ───────────────────────────────────────────
_BBH_APPROXIMANTS = frozenset([
    "IMRPhenomTPHM", "IMRPhenomXPHM", "SEOBNRv4PHM",
    "NRSur7dq4", "IMRPhenomD",
])

def _time_ms(fn):
    t0 = time.perf_counter()
    result = fn()
    return result, (time.perf_counter() - t0) * 1e3

READ_BENCHMARKS = {
    "Full file (all cols)": lambda p: pq.read_table(p),
    "injection col only":   lambda p: pq.read_table(p, columns=["injection"]),
    "row_id + injection":   lambda p: pq.read_table(p, columns=["row_id", "injection"]),
}

run_btn  = widgets.Button(description="▶  Run read benchmarks", button_style="success")
read_out = widgets.Output()
display(run_btn, read_out)

_read_results: dict = {}   # {label: {size: ms}}

def _run_reads(b):
    global _read_results
    _read_results = {label: {} for label in READ_BENCHMARKS}
    with read_out:
        clear_output(wait=True)
        print(f"{'Benchmark':<30} {'10k ms':>9} {'50k ms':>9} {'100k ms':>9} {'1M ms':>9}")
        print("─" * 60)
        for label, fn in READ_BENCHMARKS.items():
            row = f"{label:<30}"
            for s in SIZES:
                _, ms = _time_ms(lambda p=FILES[s], f=fn: f(p))
                _read_results[label][s] = ms
                row += f" {ms:>9.1f}"
            print(row)

run_btn.on_click(_run_reads)

Button(button_style='success', description='▶  Run read benchmarks', style=ButtonStyle())

Output()

## 4 · Query benchmarks

Compare PyArrow (in-memory compute) vs DuckDB (SQL) for common user queries.
Click **▶ Run queries** to populate the results table and chart below.

In [14]:
# ── 4a. PyArrow query helpers ───────────────────────────────────────────────
def _pa_json_floats(col, key):
    params = pc.struct_field(col, "parameters")
    vals   = []
    for blob in params.to_pylist():
        try:
            vals.append(json.loads(blob).get(key))
        except Exception:
            vals.append(None)
    return pa.array(vals, type=pa.float64())

def _pa_bbh_heavy(path):
    tbl = pq.read_table(path, columns=["injection"])
    col = tbl.column("injection")
    is_bbh = pc.is_in(pc.struct_field(col, "approximant"),
                      value_set=pa.array(list(_BBH_APPROXIMANTS)))
    bbh_col = tbl.filter(is_bbh).column("injection")
    m1 = _pa_json_floats(bbh_col, "mass1")
    m2 = _pa_json_floats(bbh_col, "mass2")
    return int(pc.sum(pc.greater(pc.add(m1, m2), 60.0).cast(pa.int32())).as_py())

def _pa_sg_q100(path):
    tbl = pq.read_table(path, columns=["injection"])
    col = tbl.column("injection")
    sg_col = tbl.filter(
        pc.equal(pc.struct_field(col, "approximant"), "SGE")
    ).column("injection")
    q = _pa_json_floats(sg_col, "Q")
    return int(pc.sum(pc.equal(q, 100.0).cast(pa.int32())).as_py())

def _pa_wnb_lowfreq(path):
    tbl = pq.read_table(path, columns=["injection"])
    col = tbl.column("injection")
    wnb_col = tbl.filter(
        pc.equal(pc.struct_field(col, "approximant"), "WNB")
    ).column("injection")
    freq = _pa_json_floats(wnb_col, "frequency")
    return int(pc.sum(pc.less(freq, 100.0).cast(pa.int32())).as_py())

def _pa_roundtrip(path):
    tbl = pq.read_table(path, columns=["injection"])
    return sum(1 for row in tbl.column("injection").to_pylist()
               if InjectionParams(**row) is not None)

def _pa_filter(path, approx_val):
    tbl = pq.read_table(path, columns=["injection"])
    col = tbl.column("injection")
    mask = pc.equal(pc.struct_field(col, "approximant"), approx_val)
    return len(tbl.filter(mask))

def _pa_bbh_only(path):
    tbl = pq.read_table(path, columns=["injection"])
    col = tbl.column("injection")
    mask = pc.is_in(pc.struct_field(col, "approximant"),
                    value_set=pa.array(list(_BBH_APPROXIMANTS)))
    return len(tbl.filter(mask))

def _pa_high_hrss(path, thr=1e-22):
    tbl = pq.read_table(path, columns=["injection"])
    col = tbl.column("injection")
    mask = pc.greater(pc.struct_field(col, "hrss").cast(pa.float64()), thr)
    return len(tbl.filter(mask))

def _pa_gps_window(path, ref=1187008882.4, half=43082.0):
    tbl = pq.read_table(path, columns=["injection"])
    col = tbl.column("injection")
    gps = pc.struct_field(col, "gps_time")
    mask = pc.and_(pc.greater_equal(gps, ref - half), pc.less_equal(gps, ref + half))
    return len(tbl.filter(mask))

PA_QUERIES = {
    "WNB only":          lambda p: _pa_filter(p, "WNB"),
    "SG only":           lambda p: _pa_filter(p, "SGE"),
    "BBH only":          _pa_bbh_only,
    "hrss > 1e-22":      _pa_high_hrss,
    "GPS 1-day window":  _pa_gps_window,
    "BBH total mass>60": _pa_bbh_heavy,
    "SG Q=100":          _pa_sg_q100,
    "WNB freq<100 Hz":   _pa_wnb_lowfreq,
    "Full roundtrip":    _pa_roundtrip,
}

print("PyArrow query helpers defined.")

PyArrow query helpers defined.


In [15]:
# ── 4b. DuckDB query helpers ────────────────────────────────────────────────
DUCK_QUERIES = {
    "WNB only":
        "SELECT COUNT(*) FROM t WHERE injection.approximant = 'WNB'",
    "SG only":
        "SELECT COUNT(*) FROM t WHERE injection.approximant = 'SGE'",
    "BBH only":
        "SELECT COUNT(*) FROM t WHERE injection.approximant NOT IN ('WNB','SGE')",
    "hrss > 1e-22":
        "SELECT COUNT(*) FROM t WHERE injection.hrss > 1e-22",
    "GPS 1-day window": (
        "SELECT COUNT(*) FROM t WHERE injection.gps_time "
        "BETWEEN 1187008882.4-43082 AND 1187008882.4+43082"),
    "BBH total mass>60": (
        "SELECT COUNT(*) FROM t "
        "WHERE injection.approximant NOT IN ('WNB','SGE') "
        "  AND (TRY_CAST(json_extract(injection.parameters,'$.mass1') AS FLOAT)"
        "      +TRY_CAST(json_extract(injection.parameters,'$.mass2') AS FLOAT)) > 60"),
    "SG Q=100": (
        "SELECT COUNT(*) FROM t "
        "WHERE injection.approximant='SGE' "
        "  AND TRY_CAST(json_extract(injection.parameters,'$.Q') AS FLOAT)=100.0"),
    "WNB freq<100 Hz": (
        "SELECT COUNT(*) FROM t "
        "WHERE injection.approximant='WNB' "
        "  AND TRY_CAST(json_extract(injection.parameters,'$.frequency') AS FLOAT)<100"),
    "Full roundtrip":
        "SELECT COUNT(*) FROM t",
}

def _run_duck(path: Path, sql: str) -> tuple[int, float]:
    if not HAS_DUCKDB:
        return -1, float("nan")
    con = duckdb.connect(":memory:")
    con.execute(f"CREATE VIEW t AS SELECT * FROM read_parquet('{path}')")
    t0  = time.perf_counter()
    n   = con.execute(sql).fetchone()[0]
    ms  = (time.perf_counter() - t0) * 1e3
    con.close()
    return n, ms

print("DuckDB query helpers defined.")

DuckDB query helpers defined.


In [ ]:
# ── 4c. Run all queries interactively ──────────────────────────────────────
query_btn = widgets.Button(description="▶  Run queries", button_style="success")
query_out = widgets.Output()
display(query_btn, query_out)

_results: dict[str, dict] = {}   # {size: {query: {pa_ms, duck_ms, n}}}

def _run_queries(b):
    global _results
    _results = {}
    with query_out:
        clear_output(wait=True)
        queries = list(PA_QUERIES.keys())
        for s in SIZES:
            path = FILES[s]
            _results[s] = {}
            n_rows = pq.read_metadata(path).num_rows
            print(f"\n── {s}  ({n_rows:,} rows) ──")
            print(f"  {'Query':<24} {'Rows':>8} {'PA ms':>9} {'Duck ms':>9}")
            print(f"  {'─'*24} {'─'*8} {'─'*9} {'─'*9}")
            for q in queries:
                _, pa_ms = _time_ms(lambda p=path, f=PA_QUERIES[q]: f(p))
                n, duck_ms = _run_duck(path, DUCK_QUERIES.get(q, "SELECT 0"))
                _results[s][q] = {"pa_ms": pa_ms, "duck_ms": duck_ms, "n": n}
                duck_str = f"{duck_ms:>9.1f}" if not np.isnan(duck_ms) else "      N/A"
                print(f"  {q:<24} {n:>8,} {pa_ms:>9.1f}{duck_str}")
    print("\n✓ Done — run the chart cells below.")

query_btn.on_click(_run_queries)

## 5 · Visualise results

After running the queries above, use the controls below to explore the timing data.

In [ ]:
# ── 5a. PyArrow vs DuckDB bar chart ────────────────────────────────────────
chart_size = widgets.ToggleButtons(
    options=SIZES, value="1000k",
    description="Dataset:",
    button_style="info",
)
log_toggle = widgets.Checkbox(value=True, description="Log scale")
chart_btn  = widgets.Button(description="📊  Draw bar chart", button_style="primary")
chart_out  = widgets.Output()

display(widgets.HBox([chart_size, log_toggle]), chart_btn, chart_out)

def _draw_bar(b):
    with chart_out:
        clear_output(wait=True)
        s = chart_size.value
        if s not in _results or not _results[s]:
            print("Run the queries first (cell above).")
            return
        data   = _results[s]
        labels = list(data.keys())
        pa_ms  = [data[q]["pa_ms"]   for q in labels]
        dk_ms  = [data[q]["duck_ms"] for q in labels]

        x = np.arange(len(labels))
        w = 0.38
        fig, ax = plt.subplots(figsize=(13, 5))
        ax.bar(x - w/2, pa_ms, w, label="PyArrow", color="#4C8BF5", alpha=0.88)
        ax.bar(x + w/2, dk_ms, w, label="DuckDB",  color="#F5A623", alpha=0.88)
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=35, ha="right", fontsize=9)
        ax.set_ylabel("Wall-clock time (ms)")
        ax.set_title(f"Query latency — {s} rows  |  PyArrow vs DuckDB")
        ax.legend()
        if log_toggle.value:
            ax.set_yscale("log")
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
        else:
            ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:,.0f}"))
        ax.grid(axis="y", alpha=0.3, which="both")
        plt.tight_layout()
        plt.show()

chart_btn.on_click(_draw_bar)

In [ ]:
# ── 5b. Scaling chart: timing vs dataset size for a chosen query ────────────
query_sel  = widgets.Dropdown(
    description="Query:",
    layout=widgets.Layout(width="360px"),
)
scale_btn  = widgets.Button(description="📈  Draw scaling", button_style="primary")
scale_out  = widgets.Output()

display(widgets.HBox([query_sel, scale_btn]), scale_out)

def _refresh_query_list(b=None):
    if _results:
        query_sel.options = list(next(iter(_results.values())).keys())

query_btn.on_click(lambda b: _refresh_query_list())   # auto-refresh after run

def _draw_scaling(b):
    with scale_out:
        clear_output(wait=True)
        q = query_sel.value
        if not q or not _results:
            print("Run the queries first and select a query from the dropdown.")
            return
        row_counts = {s: pq.read_metadata(FILES[s]).num_rows for s in SIZES}
        xs    = [row_counts[s] for s in SIZES if s in _results]
        pa_ys = [_results[s][q]["pa_ms"]   for s in SIZES if s in _results]
        dk_ys = [_results[s][q]["duck_ms"] for s in SIZES if s in _results]

        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(xs, pa_ys, "o-", color="#4C8BF5", linewidth=2, label="PyArrow")
        if HAS_DUCKDB:
            ax.plot(xs, dk_ys, "s-", color="#F5A623", linewidth=2, label="DuckDB")
        ax.set_xlabel("Number of rows")
        ax.set_ylabel("Wall-clock time (ms)")
        ax.set_title(f'Scaling: "{q}"')
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{int(v):,}"))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))
        ax.legend()
        ax.grid(alpha=0.3, which="both")
        plt.tight_layout()
        plt.show()

scale_btn.on_click(_draw_scaling)

In [ ]:
# ── 5c. hrss distribution by approximant ───────────────────────────────────
hist_size = widgets.ToggleButtons(
    options=SIZES, value="100k",
    description="Dataset:",
    button_style="info",
)
hist_btn = widgets.Button(description="📊  Draw hrss histogram", button_style="primary")
hist_out = widgets.Output()
display(hist_size, hist_btn, hist_out)

def _draw_hist(b):
    with hist_out:
        clear_output(wait=True)
        path   = FILES[hist_size.value]
        tbl    = pq.read_table(path, columns=["injection"])
        col    = tbl.column("injection")
        approx = pc.struct_field(col, "approximant").to_pylist()
        hrss   = pc.struct_field(col, "hrss").cast(pa.float64()).to_pylist()

        type_map = {
            "WNB": [h for a, h in zip(approx, hrss) if a == "WNB"],
            "SGE": [h for a, h in zip(approx, hrss) if a == "SGE"],
            "BBH": [h for a, h in zip(approx, hrss) if a not in ("WNB", "SGE")],
        }
        colors = {"WNB": "#4C8BF5", "SGE": "#34A853", "BBH": "#EA4335"}
        bins = np.logspace(np.log10(5e-24), np.log10(5e-19), 55)

        fig, ax = plt.subplots(figsize=(10, 4))
        for label, vals in type_map.items():
            ax.hist(vals, bins=bins, alpha=0.6, label=f"{label} ({len(vals):,})",
                    color=colors[label])
        ax.set_xscale("log")
        ax.set_xlabel("Injected hrss")
        ax.set_ylabel("Count")
        ax.set_title(f"hrss distribution by approximant — {hist_size.value}")
        ax.legend()
        ax.grid(alpha=0.3, which="both")
        plt.tight_layout()
        plt.show()

hist_btn.on_click(_draw_hist)